[link text](https://)# Week 3 Lab: Deep Feedforward Networks — From Scratch to PyTorch

**Course:** Deep Learning (COE 443) — Istinye University

**Instructor:** Asst. Prof. Dr. Yigit Bekir Kaya

**Reference:** Goodfellow, Bengio, Courville — *Deep Learning*, Chapter 6

---

## Objectives

In this lab you will:

1. **Solve the XOR problem** by hand and see why linear models fail
2. **Visualize activation functions** — ReLU, sigmoid, tanh — and understand their gradients
3. **Build an MLP from scratch** using only NumPy — forward pass, cross-entropy loss, and full backpropagation
4. **Train on MNIST** and achieve ~95% accuracy with your hand-coded network
5. **Rewrite with PyTorch** using `nn.Module` and see how autograd replaces manual gradient computation
6. **Explore key concepts** from the lecture: depth vs. width, universal approximation, and gradient flow

By the end, you will have a deep understanding of the **forward → loss → backward → update** loop and how backpropagation actually works inside neural networks.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

# For reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plot settings
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---

# Part 1: The XOR Problem — Why Linear Models Fail

**From the lecture (Slides 3–7):** A linear model computes $\hat{y} = \mathbf{x}^\top \mathbf{w} + b$, which can only produce hyperplanes. XOR is not linearly separable — no single line can separate the 0s from the 1s.

A hidden layer with ReLU activation transforms the input into a new space where the classes *become* linearly separable.

### 1.1 XOR Truth Table and Linear Model Failure

In [ ]:
# XOR dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
y_xor = np.array([0, 1, 1, 0], dtype=np.float64)

# Try to fit a linear model (logistic regression)
def sigmoid(z):
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))

# Train logistic regression on XOR
w_lin = np.random.randn(2) * 0.01
b_lin = 0.0
lr = 0.5

for _ in range(1000):
    z = X_xor @ w_lin + b_lin
    y_pred = sigmoid(z)
    error = y_pred - y_xor
    w_lin -= lr * (X_xor.T @ error) / 4
    b_lin -= lr * np.mean(error)

print("Linear model predictions on XOR:")
for x, y_true, y_p in zip(X_xor, y_xor, sigmoid(X_xor @ w_lin + b_lin)):
    print(f"  x={x}, true={y_true:.0f}, predicted={y_p:.3f} → class {int(y_p >= 0.5)}")
print("\nThe linear model outputs ~0.5 for all inputs — it CANNOT solve XOR!")

In [ ]:
# Visualize: no linear decision boundary can separate XOR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: original space
ax = axes[0]
colors = ['red' if y == 0 else 'blue' for y in y_xor]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=5, edgecolors='k', lw=2)
for i, (x, y) in enumerate(zip(X_xor, y_xor)):
    ax.annotate(f'{int(y)}', (x[0], x[1]), textcoords="offset points",
                xytext=(12, 12), fontsize=16, fontweight='bold')
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('Original x space: NOT linearly separable', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: show the linear model's decision boundary (fails)
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = sigmoid(grid @ w_lin + b_lin).reshape(xx.shape)
ax.contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.6)
ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=5, edgecolors='k', lw=2)
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('Linear model decision boundary (wrong!)', fontsize=14)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### 1.2 Solving XOR with a Hidden Layer

**From the lecture (Slide 7):** The textbook gives an exact solution using the parameters:

$$f(\mathbf{x}; \mathbf{W}, \mathbf{c}, \mathbf{w}, b) = \mathbf{w}^\top \max\{0, \mathbf{W}^\top \mathbf{x} + \mathbf{c}\} + b$$

with $\mathbf{W} = \begin{bmatrix}1 & 1 \\ 1 & 1\end{bmatrix}$, $\mathbf{c} = \begin{bmatrix}0 \\ -1\end{bmatrix}$, $\mathbf{w} = \begin{bmatrix}1 \\ -2\end{bmatrix}$, $b = 0$.

Let's verify this step by step.

In [ ]:
# Textbook XOR solution (Eq. 6.3–6.6)
W = np.array([[1, 1],
              [1, 1]], dtype=np.float64)
c = np.array([0, -1], dtype=np.float64)
w = np.array([1, -2], dtype=np.float64)
b = 0.0

print("Step-by-step XOR computation:")
print("=" * 60)

for x in X_xor:
    # Step 1: Linear transform
    a = W.T @ x + c
    # Step 2: ReLU activation
    h = np.maximum(0, a)
    # Step 3: Output
    y_out = w @ h + b

    print(f"\nx = {x}")
    print(f"  Step 1: a = W^T x + c = {a}")
    print(f"  Step 2: h = ReLU(a)   = {h}")
    print(f"  Step 3: y = w^T h + b = {y_out:.0f}")

print("\n" + "=" * 60)
print("XOR solved! The hidden layer maps inputs to a linearly separable space.")

In [ ]:
# Visualize the learned h-space
H = np.maximum(0, X_xor @ W + c)  # hidden representations

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: original space
ax = axes[0]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=5, edgecolors='k', lw=2)
for i, (x, y) in enumerate(zip(X_xor, y_xor)):
    ax.annotate(f'{int(y)}', (x[0], x[1]), textcoords="offset points",
                xytext=(12, 12), fontsize=16, fontweight='bold')
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('$x_1$', fontsize=14)
ax.set_ylabel('$x_2$', fontsize=14)
ax.set_title('Original x space', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Right: learned h-space
ax = axes[1]
ax.scatter(H[:, 0], H[:, 1], c=colors, s=200, zorder=5, edgecolors='k', lw=2)
for i, (h, y) in enumerate(zip(H, y_xor)):
    ax.annotate(f'{int(y)}', (h[0], h[1]), textcoords="offset points",
                xytext=(12, 12), fontsize=16, fontweight='bold')
# Draw the separating line in h-space
h1_line = np.linspace(-0.5, 2.5, 100)
h2_line = (1 - h1_line * w[0]) / (-w[1])  # w^T h = 0 => h2 = h1/2
ax.plot(h1_line, h2_line, 'k--', lw=2, label='Decision boundary')
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('$h_1$', fontsize=14)
ax.set_ylabel('$h_2$', fontsize=14)
ax.set_title('Learned h space: NOW linearly separable!', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: The hidden layer TRANSFORMS the input space.")
print("Points (0,0) and (1,1) both map to h=(0,0) and h=(2,1), while")
print("(0,1) and (1,0) both map to h=(1,0). Now a line can separate them!")

---

# Part 2: Activation Functions

**From the lecture (Slides 12–15):** Activation functions introduce nonlinearity. Without them, stacking linear layers is equivalent to a single linear layer.

We visualize the four key activation functions and their gradients.

In [ ]:
z = np.linspace(-5, 5, 500)

# Activation functions
relu = np.maximum(0, z)
sigmoid_vals = sigmoid(z)
tanh_vals = np.tanh(z)
leaky_relu = np.where(z > 0, z, 0.01 * z)

# Their derivatives
relu_grad = (z > 0).astype(float)
sigmoid_grad = sigmoid_vals * (1 - sigmoid_vals)
tanh_grad = 1 - tanh_vals ** 2
leaky_relu_grad = np.where(z > 0, 1, 0.01)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

activations = [
    ('ReLU', relu, relu_grad, '#0088B5'),
    ('Sigmoid', sigmoid_vals, sigmoid_grad, '#E53E3E'),
    ('Tanh', tanh_vals, tanh_grad, '#38A169'),
    ('Leaky ReLU', leaky_relu, leaky_relu_grad, '#ED8936'),
]

for i, (name, act, grad, color) in enumerate(activations):
    # Top row: activation
    axes[0, i].plot(z, act, color=color, lw=2)
    axes[0, i].axhline(0, color='gray', lw=0.5)
    axes[0, i].axvline(0, color='gray', lw=0.5)
    axes[0, i].set_title(f'{name}: g(z)', fontsize=14)
    axes[0, i].set_xlabel('z')
    axes[0, i].set_ylim(-2, 5)

    # Bottom row: gradient
    axes[1, i].plot(z, grad, color=color, lw=2)
    axes[1, i].axhline(0, color='gray', lw=0.5)
    axes[1, i].axvline(0, color='gray', lw=0.5)
    axes[1, i].set_title(f"{name}: g'(z)", fontsize=14)
    axes[1, i].set_xlabel('z')
    axes[1, i].set_ylim(-0.1, 1.2)

plt.suptitle('Activation Functions and Their Gradients', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print("  - ReLU: gradient is 1 (active) or 0 (dead) — no vanishing gradient when active")
print("  - Sigmoid: gradient max is 0.25 at z=0, vanishes for large |z|")
print("  - Tanh: gradient max is 1.0 at z=0, but still saturates for large |z|")
print("  - Leaky ReLU: small gradient (0.01) for z<0, avoids 'dead neurons'")

### 2.1 Why Nonlinearity Matters

**From the lecture (Slide 12):** Stacking linear layers without activation gives:

$$f(\mathbf{x}) = \mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = (\mathbf{W}_2 \mathbf{W}_1)\mathbf{x} + (\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2) = \mathbf{W}'\mathbf{x} + \mathbf{b}'$$

This is just one linear layer! Let's verify empirically.

In [ ]:
# Demonstrate: stacking linear layers = single linear layer
np.random.seed(0)
W1 = np.random.randn(2, 4)
b1 = np.random.randn(4)
W2 = np.random.randn(4, 3)
b2 = np.random.randn(3)

x_test = np.random.randn(2)

# Two layers without activation
h = x_test @ W1 + b1
y_two_layers = h @ W2 + b2

# Equivalent single layer
W_combined = W1 @ W2
b_combined = b1 @ W2 + b2
y_one_layer = x_test @ W_combined + b_combined

print("Two linear layers (no activation):")
print(f"  Output: {y_two_layers}")
print(f"\nEquivalent single layer:")
print(f"  Output: {y_one_layer}")
print(f"\nDifference: {np.abs(y_two_layers - y_one_layer).max():.2e}")
print("\n→ They are identical! Without nonlinearity, depth is useless.")

### 2.2 Softmax: From Logits to Probabilities

**From the lecture (Slide 15):** Softmax converts raw scores (logits) to a valid probability distribution.

In [ ]:
def softmax(z):
    """Numerically stable softmax."""
    z_shifted = z - np.max(z, axis=-1, keepdims=True)  # subtract max for stability
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)


# Example: 3-class logits
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)

print("Softmax example:")
print(f"  Logits:        {logits}")
print(f"  Probabilities: {probs}")
print(f"  Sum:           {probs.sum():.4f}")

# Show temperature effect
fig, ax = plt.subplots(figsize=(10, 5))
temperatures = [0.1, 0.5, 1.0, 2.0, 10.0]
x_pos = np.arange(3)
width = 0.15

for i, T in enumerate(temperatures):
    p = softmax(logits / T)
    ax.bar(x_pos + i * width, p, width, label=f'T={T}')

ax.set_xticks(x_pos + 2 * width)
ax.set_xticklabels(['Class 0\n(logit=2.0)', 'Class 1\n(logit=1.0)', 'Class 2\n(logit=0.1)'])
ax.set_ylabel('Probability')
ax.set_title('Softmax Temperature: Lower T → Sharper, Higher T → Uniform')
ax.legend()
plt.tight_layout()
plt.show()

---

# Part 3: Build an MLP from Scratch (NumPy)

Now we build a **2-layer MLP** (784 → 128 → 10) to classify MNIST digits using only NumPy.

We implement:
1. **Forward pass:** compute activations layer by layer
2. **Cross-entropy loss:** measure prediction error
3. **Backward pass (backpropagation):** compute gradients using the chain rule
4. **SGD update:** adjust weights in the direction of steepest descent

**Architecture:**

$$\text{Input (784)} \xrightarrow{W_1, b_1} \text{ReLU (128)} \xrightarrow{W_2, b_2} \text{Softmax (10)}$$

### 3.1 Load and Prepare MNIST

In [ ]:
# Download MNIST using torchvision, then convert to NumPy
train_dataset = datasets.MNIST(root='./data', train=True, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, download=True)

# Convert to NumPy arrays and normalize to [0, 1]
X_train_np = train_dataset.data.numpy().reshape(-1, 784).astype(np.float64) / 255.0
y_train_np = train_dataset.targets.numpy()
X_test_np = test_dataset.data.numpy().reshape(-1, 784).astype(np.float64) / 255.0
y_test_np = test_dataset.targets.numpy()

print(f"Training set: {X_train_np.shape[0]} images, each {X_train_np.shape[1]} pixels")
print(f"Test set:     {X_test_np.shape[0]} images")
print(f"Classes:      {np.unique(y_train_np)}")

# Visualize a few samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_np[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'{y_train_np[i]}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

### 3.2 Initialize Parameters

We use **He initialization** for weights (scale by $\sqrt{2/n_{\text{in}}}$) — this helps with ReLU activations by keeping the variance of activations roughly constant across layers.

In [ ]:
# Network dimensions
n_input = 784
n_hidden = 128
n_output = 10

# He initialization: scale by sqrt(2/n_in)
np.random.seed(42)
W1 = np.random.randn(n_input, n_hidden) * np.sqrt(2.0 / n_input)
b1 = np.zeros(n_hidden)
W2 = np.random.randn(n_hidden, n_output) * np.sqrt(2.0 / n_hidden)
b2 = np.zeros(n_output)

print(f"W1 shape: {W1.shape}  ({n_input}×{n_hidden} = {W1.size:,} parameters)")
print(f"b1 shape: {b1.shape}  ({n_hidden} parameters)")
print(f"W2 shape: {W2.shape}  ({n_hidden}×{n_output} = {W2.size:,} parameters)")
print(f"b2 shape: {b2.shape}  ({n_output} parameters)")
print(f"\nTotal parameters: {W1.size + b1.size + W2.size + b2.size:,}")

### 3.3 Forward Pass

**Algorithm 6.3 from the lecture (Slide 24):**

$$\mathbf{a}^{(1)} = \mathbf{X} \mathbf{W}_1 + \mathbf{b}_1$$
$$\mathbf{h}^{(1)} = \text{ReLU}(\mathbf{a}^{(1)})$$
$$\mathbf{a}^{(2)} = \mathbf{h}^{(1)} \mathbf{W}_2 + \mathbf{b}_2$$
$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{a}^{(2)})$$

In [ ]:
def relu(z):
    return np.maximum(0, z)


def forward(X, W1, b1, W2, b2):
    """Forward pass through a 2-layer MLP.

    Returns all intermediate values needed for backprop.
    """
    # Layer 1: linear + ReLU
    a1 = X @ W1 + b1          # pre-activation (m, 128)
    h1 = relu(a1)             # post-activation (m, 128)

    # Layer 2: linear + softmax
    a2 = h1 @ W2 + b2         # pre-activation (m, 10)
    y_hat = softmax(a2)       # probabilities (m, 10)

    # Store everything for backward pass
    cache = (X, a1, h1, a2, y_hat)
    return y_hat, cache


# Test forward pass on a small batch
y_hat, cache = forward(X_train_np[:5], W1, b1, W2, b2)
print("Forward pass output (first 5 samples):")
print(f"  Shape: {y_hat.shape}")
print(f"  Sum of probabilities: {y_hat.sum(axis=1)}")
print(f"  Predictions: {y_hat.argmax(axis=1)}")
print(f"  True labels:  {y_train_np[:5]}")

### 3.4 Cross-Entropy Loss

For multiclass classification with softmax outputs, the loss is:

$$J = -\frac{1}{m} \sum_{i=1}^{m} \log \hat{y}^{(i)}_{y^{(i)}}$$

This is the negative log-probability assigned to the correct class.

In [ ]:
def cross_entropy_loss(y_hat, y_true):
    """Compute cross-entropy loss.

    Args:
        y_hat: predicted probabilities, shape (m, 10)
        y_true: true labels (integers), shape (m,)

    Returns:
        Scalar loss value
    """
    m = y_true.shape[0]
    # Pick the predicted probability for the correct class
    log_probs = -np.log(y_hat[np.arange(m), y_true] + 1e-12)
    return np.mean(log_probs)


# Test
loss = cross_entropy_loss(y_hat, y_train_np[:5])
print(f"Initial loss (random weights): {loss:.4f}")
print(f"Expected for random 10-class: -log(1/10) = {-np.log(0.1):.4f}")

### 3.5 Backward Pass (Backpropagation)

**Algorithm 6.4 from the lecture (Slide 25).** We compute gradients layer by layer, from output to input.

**Key derivation:** For softmax + cross-entropy, the gradient of the loss w.r.t. the pre-softmax logits simplifies beautifully:

$$\frac{\partial J}{\partial \mathbf{a}^{(2)}} = \frac{1}{m}(\hat{\mathbf{y}} - \mathbf{y}_{\text{one-hot}})$$

Then we propagate backwards through each layer using the chain rule.

In [ ]:
def backward(y_hat, y_true, cache, W1, W2):
    """Backward pass: compute gradients for all parameters.

    Returns:
        dW1, db1, dW2, db2: gradients for each parameter
    """
    X, a1, h1, a2, _ = cache
    m = y_true.shape[0]

    # ---- Output layer gradient ----
    # For softmax + cross-entropy: dL/da2 = (y_hat - y_onehot) / m
    y_onehot = np.zeros_like(y_hat)
    y_onehot[np.arange(m), y_true] = 1.0
    delta2 = (y_hat - y_onehot) / m  # (m, 10)

    # Gradients for W2 and b2
    dW2 = h1.T @ delta2              # (128, 10)
    db2 = np.sum(delta2, axis=0)     # (10,)

    # ---- Hidden layer gradient ----
    # Propagate gradient through W2
    dh1 = delta2 @ W2.T              # (m, 128)

    # Propagate through ReLU: gradient is 1 if a1 > 0, else 0
    delta1 = dh1 * (a1 > 0)          # (m, 128) — element-wise!

    # Gradients for W1 and b1
    dW1 = X.T @ delta1               # (784, 128)
    db1 = np.sum(delta1, axis=0)     # (128,)

    return dW1, db1, dW2, db2


# Test backward pass
dW1, db1, dW2, db2 = backward(y_hat, y_train_np[:5], cache, W1, W2)
print("Gradient shapes:")
print(f"  dW1: {dW1.shape} (same as W1: {W1.shape})")
print(f"  db1: {db1.shape} (same as b1: {b1.shape})")
print(f"  dW2: {dW2.shape} (same as W2: {W2.shape})")
print(f"  db2: {db2.shape} (same as b2: {b2.shape})")

### 3.6 Gradient Check (Numerical Verification)

Before training, let's verify our gradients are correct by comparing with numerical differentiation:

$$\frac{\partial J}{\partial \theta_i} \approx \frac{J(\theta_i + \epsilon) - J(\theta_i - \epsilon)}{2\epsilon}$$

In [ ]:
def numerical_gradient(X, y, W1, b1, W2, b2, param_name, epsilon=1e-5):
    """Compute numerical gradient for one parameter matrix."""
    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    param = params[param_name]
    num_grad = np.zeros_like(param)

    it = np.nditer(param, flags=['multi_index'])
    count = 0
    while not it.finished and count < 20:  # check only first 20 elements
        idx = it.multi_index
        old_val = param[idx]

        param[idx] = old_val + epsilon
        y_hat_plus, _ = forward(X, W1, b1, W2, b2)
        loss_plus = cross_entropy_loss(y_hat_plus, y)

        param[idx] = old_val - epsilon
        y_hat_minus, _ = forward(X, W1, b1, W2, b2)
        loss_minus = cross_entropy_loss(y_hat_minus, y)

        num_grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
        param[idx] = old_val

        it.iternext()
        count += 1

    return num_grad


# Small batch for gradient check
X_small = X_train_np[:10]
y_small = y_train_np[:10]

y_hat_check, cache_check = forward(X_small, W1, b1, W2, b2)
dW1_analytic, db1_analytic, dW2_analytic, db2_analytic = backward(
    y_hat_check, y_small, cache_check, W1, W2
)

# Check W2 gradients (faster to check)
dW2_numerical = numerical_gradient(X_small, y_small, W1, b1, W2, b2, 'W2')

# Compare first 20 elements
flat_analytic = dW2_analytic.ravel()[:20]
flat_numerical = dW2_numerical.ravel()[:20]
rel_error = np.abs(flat_analytic - flat_numerical) / (np.abs(flat_analytic) + np.abs(flat_numerical) + 1e-12)

print("Gradient check (W2, first 20 elements):")
print(f"  Max relative error: {rel_error.max():.2e}")
print(f"  Mean relative error: {rel_error.mean():.2e}")
if rel_error.max() < 1e-5:
    print("  ✓ Gradients look correct!")
else:
    print("  ✗ Gradients may have a bug — check your backward pass.")

### 3.7 Training Loop with Minibatch SGD

In [ ]:
def train_numpy_mlp(X_train, y_train, X_test, y_test,
                    n_hidden=128, lr=0.1, epochs=10, batch_size=128):
    """Train a 2-layer MLP from scratch using NumPy."""
    n_input = X_train.shape[1]
    n_output = len(np.unique(y_train))
    m = X_train.shape[0]

    # Initialize weights
    np.random.seed(42)
    W1 = np.random.randn(n_input, n_hidden) * np.sqrt(2.0 / n_input)
    b1 = np.zeros(n_hidden)
    W2 = np.random.randn(n_hidden, n_output) * np.sqrt(2.0 / n_hidden)
    b2 = np.zeros(n_output)

    train_losses = []
    test_accs = []

    for epoch in range(epochs):
        # Shuffle training data
        perm = np.random.permutation(m)
        X_shuffled = X_train[perm]
        y_shuffled = y_train[perm]

        epoch_loss = 0
        n_batches = 0

        for start in range(0, m, batch_size):
            end = min(start + batch_size, m)
            X_batch = X_shuffled[start:end]
            y_batch = y_shuffled[start:end]

            # Forward
            y_hat, cache = forward(X_batch, W1, b1, W2, b2)
            loss = cross_entropy_loss(y_hat, y_batch)
            epoch_loss += loss
            n_batches += 1

            # Backward
            dW1, db1, dW2, db2 = backward(y_hat, y_batch, cache, W1, W2)

            # Update (SGD)
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2

        # Evaluate
        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)

        y_pred_test, _ = forward(X_test, W1, b1, W2, b2)
        test_acc = np.mean(y_pred_test.argmax(axis=1) == y_test)
        test_accs.append(test_acc)

        print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | Test Acc: {test_acc:.2%}")

    return W1, b1, W2, b2, train_losses, test_accs


# Train!
W1_trained, b1_trained, W2_trained, b2_trained, losses, accs = train_numpy_mlp(
    X_train_np, y_train_np, X_test_np, y_test_np,
    n_hidden=128, lr=0.1, epochs=10, batch_size=128
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(losses) + 1), losses, 'b-o', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')

axes[1].plot(range(1, len(accs) + 1), [a * 100 for a in accs], 'g-o', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].set_ylim(80, 100)

plt.suptitle('NumPy MLP on MNIST (from scratch!)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy: {accs[-1]:.2%}")
print("You implemented every gradient by hand — no autograd!")

In [ ]:
# Visualize some predictions
y_pred_final, _ = forward(X_test_np, W1_trained, b1_trained, W2_trained, b2_trained)
predictions = y_pred_final.argmax(axis=1)

# Find some correct and incorrect predictions
correct = np.where(predictions == y_test_np)[0]
incorrect = np.where(predictions != y_test_np)[0]

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Top: Correct | Bottom: Incorrect', fontsize=14)

for i, ax in enumerate(axes[0]):
    idx = correct[i]
    ax.imshow(X_test_np[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'{predictions[idx]}', color='green', fontsize=12)
    ax.axis('off')

for i, ax in enumerate(axes[1]):
    idx = incorrect[i]
    ax.imshow(X_test_np[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'{predictions[idx]} (true:{y_test_np[idx]})', color='red', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

---

# Part 4: Backpropagation Step-by-Step

**From the lecture (Slides 21–26):** Let's trace backprop through a concrete example with actual numbers, to build intuition for how gradients flow through the network.

In [ ]:
# Small network: 2 inputs → 3 hidden (ReLU) → 2 output (softmax)
np.random.seed(7)
x_demo = np.array([[0.5, -0.3]])  # single input
y_demo = np.array([1])             # true class = 1

W1_demo = np.array([[0.2, -0.5, 0.1],
                    [0.4,  0.3, -0.2]])
b1_demo = np.array([0.1, -0.1, 0.0])
W2_demo = np.array([[0.3, -0.4],
                    [-0.1, 0.5],
                    [0.2, 0.1]])
b2_demo = np.array([0.0, 0.0])

print("=" * 65)
print("FORWARD PASS (computing activations from input to output)")
print("=" * 65)

# Step 1: Layer 1 pre-activation
a1_demo = x_demo @ W1_demo + b1_demo
print(f"\n1. Pre-activation a1 = x @ W1 + b1 = {a1_demo}")

# Step 2: ReLU
h1_demo = np.maximum(0, a1_demo)
print(f"2. Post-activation h1 = ReLU(a1) = {h1_demo}")
print(f"   (Note: negative values become 0 — those neurons are 'dead')")

# Step 3: Layer 2 pre-activation
a2_demo = h1_demo @ W2_demo + b2_demo
print(f"3. Pre-activation a2 = h1 @ W2 + b2 = {a2_demo}")

# Step 4: Softmax
y_hat_demo = softmax(a2_demo)
print(f"4. Output y_hat = softmax(a2) = {y_hat_demo}")
print(f"   Predicted class: {y_hat_demo.argmax()}, True class: {y_demo[0]}")

# Step 5: Loss
loss_demo = cross_entropy_loss(y_hat_demo, y_demo)
print(f"5. Loss = -log(y_hat[true_class]) = -log({y_hat_demo[0, y_demo[0]]:.4f}) = {loss_demo:.4f}")

In [ ]:
print("=" * 65)
print("BACKWARD PASS (computing gradients from output to input)")
print("=" * 65)

# Step 1: Output gradient (softmax + cross-entropy shortcut)
y_oh = np.zeros_like(y_hat_demo)
y_oh[0, y_demo[0]] = 1.0
delta2_demo = (y_hat_demo - y_oh)  # (1, 2)
print(f"\n1. δ2 = y_hat - y_onehot = {delta2_demo}")
print(f"   (positive for wrong class, negative for correct class)")

# Step 2: Gradients for W2, b2
dW2_demo = h1_demo.T @ delta2_demo
db2_demo = delta2_demo.sum(axis=0)
print(f"\n2. dW2 = h1^T @ δ2 =\n{dW2_demo}")
print(f"   db2 = sum(δ2) = {db2_demo}")

# Step 3: Propagate to hidden layer
dh1_demo = delta2_demo @ W2_demo.T  # (1, 3)
print(f"\n3. dh1 = δ2 @ W2^T = {dh1_demo}")

# Step 4: Through ReLU
relu_mask = (a1_demo > 0).astype(float)
delta1_demo = dh1_demo * relu_mask
print(f"4. ReLU mask = {relu_mask}")
print(f"   δ1 = dh1 * mask = {delta1_demo}")
print(f"   (Gradients are zeroed where ReLU was inactive!)")

# Step 5: Gradients for W1, b1
dW1_demo = x_demo.T @ delta1_demo
db1_demo = delta1_demo.sum(axis=0)
print(f"\n5. dW1 = x^T @ δ1 =\n{dW1_demo}")
print(f"   db1 = sum(δ1) = {db1_demo}")

print(f"\n{'=' * 65}")
print("Each gradient has the same shape as its parameter — ready for SGD update!")

### 4.1 Visualize Gradient Flow

Let's verify our manual backprop against PyTorch's autograd.

In [ ]:
# Verify with PyTorch autograd
x_pt = torch.tensor(x_demo, dtype=torch.float64, requires_grad=False)
W1_pt = torch.tensor(W1_demo, dtype=torch.float64, requires_grad=True)
b1_pt = torch.tensor(b1_demo, dtype=torch.float64, requires_grad=True)
W2_pt = torch.tensor(W2_demo, dtype=torch.float64, requires_grad=True)
b2_pt = torch.tensor(b2_demo, dtype=torch.float64, requires_grad=True)
y_pt = torch.tensor(y_demo, dtype=torch.long)

# Forward
h1_pt = torch.relu(x_pt @ W1_pt + b1_pt)
logits_pt = h1_pt @ W2_pt + b2_pt
loss_pt = F.cross_entropy(logits_pt, y_pt)

# Backward
loss_pt.backward()

# Compare
print("Gradient verification (NumPy vs PyTorch autograd):")
print(f"  dW2 max diff: {np.abs(dW2_demo - W2_pt.grad.numpy()).max():.2e}")
print(f"  db2 max diff: {np.abs(db2_demo - b2_pt.grad.numpy()).max():.2e}")
print(f"  dW1 max diff: {np.abs(dW1_demo - W1_pt.grad.numpy()).max():.2e}")
print(f"  db1 max diff: {np.abs(db1_demo - b1_pt.grad.numpy()).max():.2e}")
print("\n✓ Our manual backprop matches PyTorch autograd exactly!")

---

# Part 5: MLP with PyTorch

Now we rewrite Part 3 using PyTorch. The key mapping:

| From Scratch (NumPy) | PyTorch Equivalent |
|---|---|
| `X @ W1 + b1` | `nn.Linear(784, 128)` |
| `np.maximum(0, z)` | `F.relu()` |
| `softmax(X @ W2 + b2)` | `nn.Linear(128, 10)` |
| `cross_entropy_loss()` | `nn.CrossEntropyLoss()` |
| Manual backprop | `loss.backward()` |
| `W -= lr * dW` | `optim.SGD` + `optimizer.step()` |

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_input=784, n_hidden=128, n_output=10):
        super().__init__()
        self.fc1 = nn.Linear(n_input, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_output)

    def forward(self, x):
        h = F.relu(self.fc1(x))  # Hidden layer + ReLU
        return self.fc2(h)       # Output logits (no softmax — CrossEntropyLoss handles it)


model = MLP()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Prepare data loaders
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.1307,), (0.3081,))])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=128, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=False, transform=transform),
    batch_size=1000
)

In [ ]:
# Training loop
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

pt_losses = []
pt_accs = []

for epoch in range(10):
    model.train()
    epoch_loss = 0
    n_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.view(-1, 784)  # flatten

        # Forward
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        # Backward + Update
        optimizer.zero_grad()
        loss.backward()        # autograd computes ALL gradients
        optimizer.step()       # SGD update

        epoch_loss += loss.item()
        n_batches += 1

    # Evaluate
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.view(-1, 784)
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = epoch_loss / n_batches
    test_acc = correct / total
    pt_losses.append(avg_loss)
    pt_accs.append(test_acc)

    print(f"Epoch {epoch+1:2d}/10 | Loss: {avg_loss:.4f} | Test Acc: {test_acc:.2%}")

In [ ]:
# Compare NumPy vs PyTorch results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 11), losses, 'b-o', label='NumPy (from scratch)', markersize=4)
axes[0].plot(range(1, 11), pt_losses, 'r-s', label='PyTorch (autograd)', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(range(1, 11), [a * 100 for a in accs], 'b-o', label='NumPy', markersize=4)
axes[1].plot(range(1, 11), [a * 100 for a in pt_accs], 'r-s', label='PyTorch', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].set_ylim(80, 100)
axes[1].legend()

plt.suptitle('NumPy MLP vs PyTorch MLP on MNIST', fontsize=14)
plt.tight_layout()
plt.show()

print("Both achieve similar accuracy!")
print("NumPy: 100+ lines of gradient code | PyTorch: ~20 lines total")

---

# Part 6: Exploring Key Concepts

These exercises connect the lab to the core lecture topics from Chapter 6.

### 6.1 Depth vs. Width

**From the lecture (Slides 17–18):** The Universal Approximation Theorem says one hidden layer is enough to *represent* any function, but deeper networks can be exponentially more efficient.

Let's compare shallow-wide vs. deep-narrow networks with the same number of parameters.

In [ ]:
class ShallowWide(nn.Module):
    """1 hidden layer, many units."""
    def __init__(self, width=512):
        super().__init__()
        self.fc1 = nn.Linear(784, width)
        self.fc2 = nn.Linear(width, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class DeepNarrow(nn.Module):
    """4 hidden layers, fewer units each."""
    def __init__(self, width=64):
        super().__init__()
        self.fc1 = nn.Linear(784, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, width)
        self.fc4 = nn.Linear(width, width)
        self.fc5 = nn.Linear(width, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        return self.fc5(x)


def train_and_eval(model, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    results = []

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.view(-1, 784)
            loss = criterion(model(X_batch), y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.view(-1, 784)
                correct += (model(X_batch).argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)
        results.append(correct / total)

    return results


torch.manual_seed(42)
shallow = ShallowWide(512)
deep = DeepNarrow(64)

n_params_shallow = sum(p.numel() for p in shallow.parameters())
n_params_deep = sum(p.numel() for p in deep.parameters())

print(f"Shallow (1 hidden, width=512): {n_params_shallow:,} parameters")
print(f"Deep    (4 hidden, width=64):  {n_params_deep:,} parameters")

print("\nTraining shallow model...")
accs_shallow = train_and_eval(shallow, epochs=5)
print("Training deep model...")
accs_deep = train_and_eval(deep, epochs=5)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 6), [a * 100 for a in accs_shallow], 'b-o',
         label=f'Shallow (1 layer, {n_params_shallow:,} params)')
plt.plot(range(1, 6), [a * 100 for a in accs_deep], 'r-s',
         label=f'Deep (4 layers, {n_params_deep:,} params)')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy (%)')
plt.title('Depth vs Width on MNIST')
plt.legend()
plt.ylim(90, 100)
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nShallow final: {accs_shallow[-1]:.2%}")
print(f"Deep final:    {accs_deep[-1]:.2%}")

### 6.2 Cross-Entropy vs. MSE for Classification

**From the lecture (Slide 9):** Cross-entropy gives strong gradients when the model is wrong, while MSE with sigmoid saturates. Let's see this in practice.

In [ ]:
# Compare cross-entropy vs MSE loss surfaces for sigmoid output
z = np.linspace(-6, 6, 500)
sigma = sigmoid(z)

# Target = 1
ce_loss = -np.log(sigma + 1e-12)              # cross-entropy
mse_loss = (sigma - 1) ** 2                    # MSE

# Gradients w.r.t. z
ce_grad = sigma - 1                            # dL/dz for CE = sigma - y
mse_grad = 2 * (sigma - 1) * sigma * (1 - sigma)  # dL/dz for MSE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(z, ce_loss, label='Cross-Entropy', lw=2)
axes[0].plot(z, mse_loss, label='MSE', lw=2)
axes[0].set_xlabel('z (logit)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (target = 1)')
axes[0].legend()
axes[0].set_ylim(0, 4)

axes[1].plot(z, np.abs(ce_grad), label='|dL/dz| Cross-Entropy', lw=2)
axes[1].plot(z, np.abs(mse_grad), label='|dL/dz| MSE', lw=2)
axes[1].set_xlabel('z (logit)')
axes[1].set_ylabel('|Gradient|')
axes[1].set_title('Gradient Magnitude (target = 1)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - When z << 0 (model is very wrong), CE gradient is LARGE → fast correction")
print("  - When z << 0, MSE gradient is TINY → sigmoid saturates → learning stalls")
print("  - This is why we always use cross-entropy for classification!")

### 6.3 Visualizing Learned Representations

What does the hidden layer actually learn? Let's extract the hidden representations from our trained PyTorch model and visualize them.

In [ ]:
# Extract hidden representations from the trained PyTorch model
from sklearn.decomposition import PCA

model.eval()
hidden_reps = []
labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.view(-1, 784)
        h = F.relu(model.fc1(X_batch))  # hidden layer output
        hidden_reps.append(h.numpy())
        labels.append(y_batch.numpy())

hidden_reps = np.concatenate(hidden_reps)
labels = np.concatenate(labels)

# PCA to 2D for visualization
pca = PCA(n_components=2)
hidden_2d = pca.fit_transform(hidden_reps)

plt.figure(figsize=(10, 8))
for digit in range(10):
    mask = labels == digit
    plt.scatter(hidden_2d[mask, 0], hidden_2d[mask, 1],
                alpha=0.3, s=5, label=str(digit))
plt.legend(markerscale=5, fontsize=10)
plt.title('Hidden Layer Representations (PCA → 2D)', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print("The network has learned to CLUSTER digits in its hidden space!")
print("Similar digits are close together, different digits are far apart.")
print("This is the 'learned representation' that makes classification easy.")

### 6.4 The Effect of Width on Learning

How does the number of hidden units affect what the network can learn?

In [ ]:
widths = [8, 32, 128, 512]
results_width = {}

for w in widths:
    torch.manual_seed(42)
    m = MLP(n_hidden=w)
    n_params = sum(p.numel() for p in m.parameters())
    acc_list = train_and_eval(m, epochs=5)
    results_width[w] = (acc_list, n_params)
    print(f"Width={w:4d} | Params={n_params:>8,} | Final acc: {acc_list[-1]:.2%}")

plt.figure(figsize=(10, 5))
for w in widths:
    acc_list, n_params = results_width[w]
    plt.plot(range(1, 6), [a * 100 for a in acc_list],
             '-o', markersize=4, label=f'width={w} ({n_params:,} params)')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy (%)')
plt.title('Effect of Hidden Layer Width')
plt.legend()
plt.ylim(80, 100)
plt.grid(True, alpha=0.3)
plt.show()

print("\nObservation: More width → more capacity → better accuracy")
print("But diminishing returns: 128→512 gains less than 8→32")

### 6.5 PyTorch Autograd: Under the Hood

**From the lecture (Slide 27):** PyTorch builds a dynamic computation graph during the forward pass, then walks it backward to compute gradients.

In [ ]:
# Demonstrate autograd step by step
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Build computation graph
z = w * x + b        # linear
a = torch.relu(z)    # activation
loss = (a - 5) ** 2  # MSE loss with target=5

print("Computation graph:")
print(f"  x={x.item()}, w={w.item()}, b={b.item()}")
print(f"  z = w*x + b = {z.item()}")
print(f"  a = ReLU(z) = {a.item()}")
print(f"  loss = (a-5)^2 = {loss.item()}")

# Backward
loss.backward()

print(f"\nAutograd gradients:")
print(f"  dloss/dw = {w.grad.item():.4f}")
print(f"  dloss/db = {b.grad.item():.4f}")
print(f"  dloss/dx = {x.grad.item():.4f}")

# Manual verification:
# dloss/da = 2(a-5) = 2(7-5) = 4
# da/dz = 1 (since z>0, ReLU grad = 1)
# dz/dw = x = 2
# dz/db = 1
# dz/dx = w = 3
# dloss/dw = dloss/da * da/dz * dz/dw = 4 * 1 * 2 = 8
print(f"\nManual chain rule verification:")
print(f"  dloss/da = 2(a-5) = {2*(a.item()-5)}")
print(f"  da/dz = 1 (ReLU active)")
print(f"  dloss/dw = 4 * 1 * x = {4 * x.item()}  ✓")
print(f"  dloss/db = 4 * 1 * 1 = 4.0  ✓")
print(f"  dloss/dx = 4 * 1 * w = {4 * w.item()}  ✓")

---

# Summary

In this lab you:

| Part | What You Did | Key Concept |
|------|-------------|-------------|
| 1 | Solved XOR by hand | Why linear models fail, feature transformation |
| 2 | Visualized activation functions | ReLU, sigmoid, tanh, softmax and their gradients |
| 3 | Built an MLP from scratch | Forward pass, cross-entropy, backprop, SGD |
| 4 | Traced backprop step-by-step | Chain rule applied layer by layer |
| 5 | Rewrote in PyTorch | `nn.Module`, autograd, same accuracy in 5× less code |
| 6 | Explored architecture concepts | Depth vs. width, CE vs. MSE, learned representations |

**The core loop:** `forward → loss → backward → update`

You now understand what happens *inside* `loss.backward()` — it's the chain rule applied systematically to a computational graph, computing all gradients in one pass.

---

# Homework Exercises

### Exercise 1: Add a Third Layer
Extend the NumPy MLP in Part 3 to have **two hidden layers** (784 → 128 → 64 → 10). Derive and implement the backpropagation for three layers. Compare accuracy with the 2-layer version.

### Exercise 2: Activation Function Comparison
Modify the PyTorch model in Part 5 to use different activation functions (sigmoid, tanh, Leaky ReLU) in the hidden layer. Train each for 10 epochs and plot all learning curves on one graph. Which activation works best? Why?

### Exercise 3: Batch Normalization
Add `nn.BatchNorm1d` after the first linear layer (before activation) in the PyTorch model. Does it improve training speed or final accuracy? Why might this help?

### Conceptual Questions
1. In Part 1, why does ReLU activation enable the network to solve XOR while a linear activation would not? What would happen if we used sigmoid instead of ReLU in the hidden layer?
2. In Part 3, we used He initialization (`sqrt(2/n_in)`). What would happen if we initialized all weights to zero? To very large values? Why does initialization matter?
3. In Part 4, we saw that the ReLU gradient is exactly 0 for inactive neurons. What happens if a neuron gets a large negative bias during training and never activates again? This is called the "dying ReLU" problem — how does Leaky ReLU address it?

---

### Next Week

**Week 4: Regularization for Deep Learning (Ch 7)**
- Parameter norm penalties (L1, L2)
- Dropout and its probabilistic interpretation
- Data augmentation, noise robustness
- Early stopping, batch normalization
- Ensemble methods (bagging, boosting)